# 03 · Join Sofascore + Capology — Spain La Liga 24/25

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2024/25 de La Liga española**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [102]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [103]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [104]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2425.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2425.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  589 jugadores | 116 columnas
Capology:   551 jugadores | 9 columnas


## 4. Normalización

In [105]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [106]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   deportivo alaves
   girona fc
   real valladolid

En Capology pero no en Sofascore:
   alaves
   girona
   valladolid


### 5.2 Aplicar TEAM_MAP

In [107]:
TEAM_MAP = {
    'alaves'    : 'deportivo alaves',
    'girona'    : 'girona fc',
    'valladolid': 'real valladolid'
}

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [108]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 486/589 (82.5%)
Sin emparejar: 103


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [109]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          11
Revisión media    (0.75 ≤ score < 0.90):   12
Revisión estricta (0.50 ≤ score < 0.75):   49
Revisión muy est. (score < 0.50):           31


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [110]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
6,Alexander Sørloth,Atlético Madrid,alexander sorloth,0.970
91,Rafael Bauzà,Espanyol,rafel bauza,0.957
5,Viktor Tsygankov,Girona FC,viktor tsyhankov,0.938
48,Javier Hernández,Leganés,javi hernandez,0.933
38,Manuel Sánchez,Deportivo Alavés,manu sanchez,0.923
21,Manuel Fuster,Las Palmas,manu fuster,0.917
33,Javier Guerra,Valencia,javi guerra,0.917
4,Daniel Vivian,Athletic Club,dani vivian,0.917
13,Yéremy Pino,Villarreal,yeremi pino,0.909
23,Javier Muñoz,Las Palmas,javi munoz,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [111]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
18,Josep Chavarría,Rayo Vallecano,pep chavarria,0.857
54,Mario Maroto,Real Valladolid,mario martin,0.833
17,Abdessamad Ezzalzouli,Real Betis,abde ezzalzouli,0.833
25,Francisco Vieites,Real Betis,fran vieites,0.828
31,Abdelkabir Abqar,Deportivo Alavés,abdel abqar,0.815
52,Chuky,Real Valladolid,chuki,0.800
9,José María Giménez,Atlético Madrid,jose gimenez,0.800
61,Orri Steinn Óskarsson,Real Sociedad,orri oskarsson,0.800
63,Mateu Morey,Mallorca,mateu morey bauza,0.786
10,José Luis Gayà,Valencia,jose gaya,0.783


In [112]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [
    'mario maroto'
]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 11 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [113]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
22,Pathé Ismaël Ciss,Rayo Vallecano,pathe ciss,0.741
39,Raúl García de Haro,Osasuna,raul garcia,0.733
26,Urko González,Espanyol,urko gonzalez de zarate,0.722
1,Alejandro Baena,Villarreal,alex baena,0.720
19,Flavien Boyomo,Osasuna,enzo boyomo,0.720
89,Arturo Rodríguez,Las Palmas,kirian rodriguez,0.688
90,José Carlo González,Las Palmas,fabio gonzalez,0.667
83,Adrián Arnu,Real Valladolid,adam aznou,0.667
0,Sergio Arribas,Real Betis,sergi altimira,0.643
53,Gonzalo García,Real Madrid,fran garcia,0.640


In [114]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = [
    'pathe ismael ciss',
    'raul garcia de haro',
    'urko gonzalez',
    'alejandro baena',
    'flavien boyomo',
    'johnny cardoso',
    'pablo gavi',
    'rodri sanchez',
    'peter gonzalez'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 9


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [115]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
34,Alvaro Garcia-Pascual,Sevilla,alvaro fernandez,0.486
87,Aboubacar Bassinga,Las Palmas,fabio silva,0.483
69,Ismael Bekhoucha,Getafe,juan berrocal,0.483
8,Iker Almena,Girona FC,yaser asprilla,0.480
37,Peio Canales,Athletic Club,aitor paredes,0.480
78,Gorka Rivera,Getafe,borja mayoral,0.480
50,Yoel Lago,Celta Vigo,joseph aidoo,0.476
71,Dani Díaz,Real Sociedad,brais mendez,0.476
98,Rodrigo Abajas,Valencia,giorgi mamardashvili,0.471
80,Nobel Mendy,Real Betis,antony,0.471


In [116]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [
    # 'player_norm'
]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [117]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 517/589 (87.8%)
Sin salario:     72


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [118]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

print(f'Total sin salario: {len(sin_salario)}')

Total sin salario: 72


In [119]:
pd.set_option('display.max_rows', None)
sin_salario

,player,team,minutesPlayed,appearances,goals,assists
0,Peio Canales,Athletic Club,259,8,0,0
1,Aingeru Olabarrieta,Athletic Club,55,1,0,0
2,Malcom Ares,Athletic Club,18,1,0,0
3,Endika Buján,Athletic Club,10,1,0,0
4,Adrián Niño,Atlético Madrid,11,1,0,0
5,Sergi Domínguez,Barcelona,173,3,0,0
6,Dani Rodríguez,Barcelona,38,1,0,0
7,Fer López,Celta Vigo,690,17,2,0
8,Yoel Lago,Celta Vigo,583,8,0,1
9,Roger Hinojo,Espanyol,8,1,0,0


In [120]:
# Comparación manual por equipo: jugadores SF sin salario vs plantilla CG
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')
    
    equipo_norm  = normalize(equipo)
    cg_jugadores = df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']].sort_values('player_norm').reset_index(drop=True)
    
    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Athletic Club  —  SF sin salario:


,player,minutesPlayed
0,Aingeru Olabarrieta,55
1,Endika Buján,10
2,Malcom Ares,18
3,Peio Canales,259


  CG plantilla completa:


,player,player_norm
0,Adama Boiro,adama boiro
1,Aitor Paredes,aitor paredes
2,Álex Berenguer,alex berenguer
3,Álex Padilla,alex padilla
4,Álvaro Djaló,alvaro djalo
5,Ander Herrera,ander herrera
6,Andoni Gorosabel,andoni gorosabel
7,Beñat Prados,benat prados
8,Dani Vivian,dani vivian
9,Gorka Guruzeta,gorka guruzeta



  Atlético Madrid  —  SF sin salario:


...

,player,player_norm
0,André Almeida,andre almeida
1,César Tárrega,cesar tarrega
2,Cristhian Mosquera,cristhian mosquera
3,Dani Gómez,dani gomez
4,Diego López,diego lopez
5,Dimitri Foulquier,dimitri foulquier
6,Enzo Barrenechea,enzo barrenechea
7,Fran Pérez,fran perez
8,Germán Valera,german valera
9,Giorgi Mamardashvili,giorgi mamardashvili



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Arnau Solà,12
1,Dani Requena,9
2,Etta Eyong,46
3,Thiago Ojeda,13


  CG plantilla completa:


,player,player_norm
0,Álex Baena,alex baena
1,Alfonso Pedraza,alfonso pedraza
2,Ayoze Pérez,ayoze perez
3,Dani Parejo,dani parejo
4,Denis Suárez,denis suarez
5,Diego Conde,diego conde
6,Eric Bailly,eric bailly
7,Gerard Moreno,gerard moreno
8,Iker Álvarez,iker alvarez
9,Ilias Akhomach,ilias akhomach


In [121]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [122]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2425.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2425.csv
   Jugadores totales:  589
   Con salario:        517
   Sin salario (NaN):  72
   Columnas:           121
